# Chapter 13 — Correctness (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch13_correctness_python.ipynb)

This notebook goes with chapter 13 «Correctness» of the lecture notes (volume 4, *Analysis and recursion*) and with its [page of the site](https://fabiofurini.github.io/algorithms/analysis/13-correctness/). It contains the **Python** code of the algorithms with pseudo-code of the chapter: `FindMaximum`, `InsertionSort`, `BinarySearch`, `EuclideanAlgorithm`. For each algorithm the code **prints the loop invariant every time the condition is evaluated** (before the loop and after every iteration) and checks that it holds: the tables are those of the chapter. Checking an example does not prove correctness (that is done with initialization, maintenance and termination), but it shows the invariant at work.

**How to run it.** Click the «Open in Colab» badge, then *Runtime ▸ Run all* (or cell by cell with Shift+Enter). Nothing to install.

Every algorithm is a Python function; the run cells execute it on the same examples as the lecture notes and print the results.

**Indices.** In the pseudo-code the arrays are indexed from 1; in the Python code from 0: the element `a[i]` of the pseudo-code is `a[i-1]` in the code (and a loop `while i <= n` from 1 becomes `while i < n` from 0). The printed tables use the indices of the pseudo-code.

## 1. Find maximum: FindMaximum

**Input:** an array $a[1..n]$ of $n \ge 1$ integer numbers. **Output:** the maximum of $a$. **Invariant:** every time the condition is evaluated, $max = \max\{a[1],\dots,a[i-1]\}$.

Pseudo-code: section 13.2 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [1]:
def print_invariant(a, i, max_val):
    """Invariant: max = maximum of a[1..i-1]."""
    sub = a[:i]                                    # a[1..i-1] of the pseudo-code
    ok = (max_val == max(sub))
    print("%2d | %3d | %-26s | %14d | %s" % (i + 1, max_val, sub, max(sub), "ok" if ok else "FALSE"))


def find_maximum(a):
    n = len(a)
    max_val = a[0]
    i = 1                                          # i := 2
    print_invariant(a, i, max_val)                 # before the first evaluation of the condition
    while i < n:                                   # i <= n
        if a[i] > max_val:
            max_val = a[i]
        i = i + 1
        print_invariant(a, i, max_val)             # after every iteration, i.e. before the next evaluation
    return max_val

In [2]:
a = [9, 12, 3, 16, 19, 11, 12]      # example of the lecture notes
print("a = %s, n = %d" % (a, len(a)))
print(" i | max | a[1..i-1]                  | max{a[1..i-1]} | invariant")
m = find_maximum(a)
print("The loop ends with i = n + 1 = %d: FindMaximum returns %d" % (len(a) + 1, m))

a = [9, 12, 3, 16, 19, 11, 12], n = 7
 i | max | a[1..i-1]                  | max{a[1..i-1]} | invariant
 2 |   9 | [9]                        |              9 | ok
 3 |  12 | [9, 12]                    |             12 | ok
 4 |  12 | [9, 12, 3]                 |             12 | ok
 5 |  16 | [9, 12, 3, 16]             |             16 | ok
 6 |  19 | [9, 12, 3, 16, 19]         |             19 | ok
 7 |  19 | [9, 12, 3, 16, 19, 11]     |             19 | ok
 8 |  19 | [9, 12, 3, 16, 19, 11, 12] |             19 | ok
The loop ends with i = n + 1 = 8: FindMaximum returns 19


**Try it yourself.** Change the array: the invariant holds on every row.

In [3]:
a = [4, 8, 8, 1, 10, 2]      # <-- change the array
print(" i | max | a[1..i-1]                  | max{a[1..i-1]} | invariant")
m = find_maximum(a)
print("returns %d" % m)

 i | max | a[1..i-1]                  | max{a[1..i-1]} | invariant
 2 |   4 | [4]                        |              4 | ok
 3 |   8 | [4, 8]                     |              8 | ok
 4 |   8 | [4, 8, 8]                  |              8 | ok
 5 |   8 | [4, 8, 8, 1]               |              8 | ok
 6 |  10 | [4, 8, 8, 1, 10]           |             10 | ok
 7 |  10 | [4, 8, 8, 1, 10, 2]        |             10 | ok
returns 10


## 2. Insertion sort: InsertionSort

**Input:** an array $a[1..n]$ of $n \ge 1$ natural numbers. **Output:** the array sorted in non-decreasing order. **Outer invariant:** $a[1..i-1]$ contains the original elements of $a[1..i-1]$ sorted and $a[i..n]$ equals the original. **Inner invariant** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ and these elements are $> key$.

Pseudo-code: section 13.3 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [4]:
def split(a, i):
    """The array with a bar after the sorted part a[1..i-1]."""
    return "[" + ", ".join(str(x) for x in a[:i]) + " | " + ", ".join(str(x) for x in a[i:]) + "]"


def print_outer(a, orig, i):
    """Outer invariant: a[1..i-1] = original elements of a[1..i-1] sorted, a[i..n] = original."""
    ok = a[:i] == sorted(orig[:i]) and a[i:] == orig[i:]
    print("%2d | %-26s | %-19s | %s" % (i + 1, split(a, i), orig[:i], "ok" if ok else "FALSE"))


def print_inner(a, b, i, j, key):
    """Inner invariant (I1)-(I3); b = a at the start of the iteration."""
    ip, jp = i + 1, j + 1                          # indices of the pseudo-code
    left, right = a[:jp], a[jp + 1:ip]             # a[1..j] and a[j+2..i]
    ok = (left == b[:jp] and right == b[jp:ip - 1] and all(x > key for x in right))
    if j >= 0:
        cond = "a[%d] = %d %s %d: %s" % (jp, a[j], ">" if a[j] > key else "<=", key, "true" if a[j] > key else "false")
    else:
        cond = "j = 0: false"
    print("%d | %-15s | %-12s | %-9s | %-20s | %s" % (jp, a, left, right, cond, "ok" if ok else "FALSE"))


def insertion_sort(a, show_outer=True, trace_inner_at=0):
    n = len(a)
    orig = a[:]                                    # copy of the input: it is only needed to check the invariant
    i = 1                                          # i := 2
    if show_outer:
        print_outer(a, orig, i)
    while i < n:                                   # i <= n
        b = a[:]                                   # content of a at the start of the iteration (only for the inner invariant)
        key = a[i]
        j = i - 1
        tr = (i + 1 == trace_inner_at)
        if tr:
            print_inner(a, b, i, j, key)
        while j >= 0 and a[j] > key:               # j >= 1 and a[j] > key
            a[j + 1] = a[j]
            j = j - 1
            if tr:
                print_inner(a, b, i, j, key)
        a[j + 1] = key
        i = i + 1
        if show_outer:
            print_outer(a, orig, i)

In [5]:
# The outer loop: a[1..5] = (5, 2, 4, 1, 3), example of the lecture notes
a = [5, 2, 4, 1, 3]
print("a = %s" % a)
print(" i | a (sorted | rest)          | original a[1..i-1]  | invariant")
insertion_sort(a)
print("sorted: %s" % a)

# The inner loop at iteration i = 5, with key = 3 (a = (1, 2, 4, 5, 3) at the start)
print()
a = [5, 2, 4, 1, 3]
print("j | a               | a[1..j]      | a[j+2..i] | condition            | invariant")
insertion_sort(a, False, 5)
print("after line 10 (a[j+1] = key): a = %s" % a)

a = [5, 2, 4, 1, 3]
 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
sorted: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condition            | invariant
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
after line 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Try it yourself.** Change the array: every row of the outer table must stay `ok`.

In [6]:
a = [6, 3, 8, 1, 9, 2]      # <-- change the array
print(" i | a (sorted | rest)          | original a[1..i-1]  | invariant")
insertion_sort(a)

 i | a (sorted | rest)          | original a[1..i-1]  | invariant
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


## 3. Binary search: BinarySearch

**Input:** an array $a[1..n]$ sorted in non-decreasing order and a value $m$. **Output:** an index $k$ with $a[k] = m$ if $m$ is in $a$, $-1$ otherwise. **Invariant:** (I1) $1 \le left$ and $right \le n$; (I2) if $m$ is in $a$, then $m$ is in $a[left..right]$.

Pseudo-code: section 13.4 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [7]:
def print_row(a, m, left, right, mid):
    """Invariant (I1) 1 <= left and right <= n, (I2) if m is in a then it is in a[left..right]; mid = -1: empty interval."""
    sub = a[left:right + 1]
    ok = (left + 1 >= 1 and right + 1 <= len(a) and (m not in a or m in sub))
    if mid < 0:
        action = "left > right => return -1"
    elif a[mid] == m:
        action = "mid = %d, a[%d] = %d = %d => return %d" % (mid + 1, mid + 1, a[mid], m, mid + 1)
    elif a[mid] < m:
        action = "mid = %d, a[%d] = %d < %d => left = %d" % (mid + 1, mid + 1, a[mid], m, mid + 2)
    else:
        action = "mid = %d, a[%d] = %d > %d => right = %d" % (mid + 1, mid + 1, a[mid], m, mid)
    print("%4d | %5d | %-27s | %-11s | %s" % (left + 1, right + 1, sub, "ok" if ok else "FALSE", action))


def binary_search(a, m, trace=False):
    n = len(a)
    left = 0                                       # left := 1
    right = n - 1                                  # right := n
    while left <= right:
        mid = left + (right - left) // 2
        if trace:
            print_row(a, m, left, right, mid)      # invariant row for this evaluation
        if a[mid] == m:
            return mid
        if a[mid] < m:
            left = mid + 1
        else:
            right = mid - 1
    if trace:
        print_row(a, m, left, right, -1)
    return -1

In [8]:
a = [3, 9, 11, 12, 12, 16, 19]      # example of the lecture notes
print("a = %s, n = %d" % (a, len(a)))

m = 10
print()
print("m = %d" % m)
print("left | right | a[left..right]              | invariant   | action")
k = binary_search(a, m, True)
print("m = %d not found: it returns %d" % (m, k))

m = 12
print()
print("m = %d" % m)
print("left | right | a[left..right]              | invariant   | action")
k = binary_search(a, m, True)
print("m = %d found: index %d in the code (position %d in the pseudo-code)" % (m, k, k + 1))

# Exhaustive check for m = 0, ..., 20
errors = 0
for m in range(0, 21):
    k = binary_search(a, m)
    if (m in a and (k < 0 or a[k] != m)) or (m not in a and k != -1):
        errors += 1
print()
print("Check for m = 0, ..., 20: %d errors" % errors)

a = [3, 9, 11, 12, 12, 16, 19], n = 7

m = 10
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 > 10 => right = 3
   1 |     3 | [3, 9, 11]                  | ok          | mid = 2, a[2] = 9 < 10 => left = 3
   3 |     3 | [11]                        | ok          | mid = 3, a[3] = 11 > 10 => right = 2
   3 |     2 | []                          | ok          | left > right => return -1
m = 10 not found: it returns -1

m = 12
left | right | a[left..right]              | invariant   | action
   1 |     7 | [3, 9, 11, 12, 12, 16, 19]  | ok          | mid = 4, a[4] = 12 = 12 => return 4
m = 12 found: index 3 in the code (position 4 in the pseudo-code)

Check for m = 0, ..., 20: 0 errors


**Try it yourself.** Change the array (it must stay sorted) and the target value.

In [9]:
a = [2, 4, 4, 7, 9, 15, 21, 30]      # <-- change the array (sorted!)
m = 15                               # <-- and the target value
print("left | right | a[left..right]              | invariant   | action")
k = binary_search(a, m, True)
print("returns %d" % k)

left | right | a[left..right]              | invariant   | action
   1 |     8 | [2, 4, 4, 7, 9, 15, 21, 30] | ok          | mid = 4, a[4] = 7 < 15 => left = 5
   5 |     8 | [9, 15, 21, 30]             | ok          | mid = 6, a[6] = 15 = 15 => return 6
returns 5


## 4. Euclidean algorithm: EuclideanAlgorithm

**Input:** two natural numbers $n, m \ge 1$. **Output:** $\gcd(n, m)$. **Invariant:** $n \ge 1$, $m \ge 0$ and $\gcd(n,m) = \gcd(n_0,m_0)$, where $n_0, m_0$ are the input values.

Pseudo-code: section 13.5 of the [chapter page](https://fabiofurini.github.io/algorithms/analysis/13-correctness/).

In [10]:
def gcd_check(n, m):                               # gcd by enumeration of the divisors, independent of the algorithm
    if m == 0:
        return n
    d = min(n, m)
    while n % d != 0 or m % d != 0:
        d = d - 1
    return d


def print_row(n, m, g0):
    """Invariant: n >= 1, m >= 0 and gcd(n, m) = gcd(n0, m0)."""
    g = gcd_check(n, m)
    ok = (n >= 1 and m >= 0 and g == g0)
    print("%4d | %4d | %9d | %s" % (n, m, g, "ok" if ok else "FALSE"))


def euclidean_algorithm(n, m):
    g0 = gcd_check(n, m)                           # gcd(n0, m0), only to check the invariant
    print_row(n, m, g0)
    while m != 0:
        r = n % m
        n = m
        m = r
        print_row(n, m, g0)
    return n

In [11]:
n, m = 252, 105      # example of the lecture notes
print("(n0, m0) = (%d, %d)" % (n, m))
print("   n |    m | gcd(n, m) | invariant")
g = euclidean_algorithm(n, m)
print("The loop ends with m = 0: it returns n = %d" % g)

print()
n, m = 48, 18
print("(n0, m0) = (%d, %d)" % (n, m))
print("   n |    m | gcd(n, m) | invariant")
g = euclidean_algorithm(n, m)
print("The loop ends with m = 0: it returns n = %d" % g)

(n0, m0) = (252, 105)
   n |    m | gcd(n, m) | invariant
 252 |  105 |        21 | ok
 105 |   42 |        21 | ok
  42 |   21 |        21 | ok
  21 |    0 |        21 | ok
The loop ends with m = 0: it returns n = 21

(n0, m0) = (48, 18)
   n |    m | gcd(n, m) | invariant
  48 |   18 |         6 | ok
  18 |   12 |         6 | ok
  12 |    6 |         6 | ok
   6 |    0 |         6 | ok
The loop ends with m = 0: it returns n = 6


**Try it yourself.** Change the two numbers: the $\gcd$ stays constant on every row.

In [12]:
n, m = 1071, 462      # <-- change the two numbers
print("   n |    m | gcd(n, m) | invariant")
g = euclidean_algorithm(n, m)
print("returns %d" % g)

   n |    m | gcd(n, m) | invariant
1071 |  462 |        21 | ok
 462 |  147 |        21 | ok
 147 |   21 |        21 | ok
  21 |    0 |        21 | ok
returns 21
